In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from rasterstats import zonal_stats
import rasterio

FARMS_PATH = "GeoFiles/CSB/CSB_AZ_cleaned.shp"
RASTER_DIR = "GeoFiles/Precipitation_800m/Reprojected/"  
OUT_DIR    = "GeoFiles/Precipitation_800m/FarmWise/"

YEARS = range(2019, 2025)  # 2019–2024

gdf = gpd.read_file(FARMS_PATH)
print(f"Loaded {len(gdf)} farms")

for year in YEARS:
    raster_path = f"{RASTER_DIR}prism_ppt_us_30s_{year}.tif"
    out_path    = f"{OUT_DIR}Farm_Precipitation_{year}.shp"

    print(f"\nProcessing year {year} ...")

    # Read raster nodata + valid pixels
    with rasterio.open(raster_path) as src:
        nodata = src.nodata
        arr = src.read(1)
        transform = src.transform

        # Get valid pixel coordinates and values for nearest neighbor search
        rows, cols = np.where(arr != nodata)
        if len(rows) == 0:
            raise ValueError(f"No valid pixels found in {raster_path}")
        valid_coords = np.array(rasterio.transform.xy(transform, rows, cols)).T
        valid_values = arr[rows, cols]

    # Run zonal stats (mean precipitation in mm) for each farm
    stats = zonal_stats(
        gdf,
        raster_path,
        stats=["mean"],
        nodata=nodata,
        all_touched=True
    )
    df_stats = pd.DataFrame(stats)
    df_stats.index = gdf.index

    # Attach results to GeoDataFrame
    gdf[f"Preci_{year}"] = df_stats["mean"]

    # Rescue NaNs using nearest valid pixel
    nan_idx = gdf[gdf[f"Preci_{year}"].isna()].index

    # Find nearest valid pixel to farm centroid for each NaN farm
    for idx in nan_idx:
        centroid = gdf.loc[idx].geometry.centroid
        cx, cy = centroid.x, centroid.y
        d2 = (valid_coords[:,0] - cx)**2 + (valid_coords[:,1] - cy)**2
        nearest_idx = np.argmin(d2)
        gdf.at[idx, f"Preci_{year}"] = valid_values[nearest_idx]

    # Categorizing the Preci_year values
    bins = [-1e9, 250, 275, 300, 350, 1e9]  # mm
    labels = ["Very Low","Low","Medium","High","Very High"]
    gdf[f"PrecC_{year}"] = pd.cut(gdf[f"Preci_{year}"], bins=bins, labels=labels).astype(str)

    nan_after = gdf[f"Preci_{year}"].isna().sum()
    print(f"NaN farms after rescue: {nan_after}")

    gdf = gdf[['CSBID', 'CSBACRES', 'CNTY', 'geometry', f'Preci_{year}', f'PrecC_{year}']].copy()
    # Save shapefile for this year
    gdf.to_file(out_path, driver="ESRI Shapefile")
    print(f"Saved {out_path}")

print("\nAll done!")

In [ ]:
#print values and counts of PrecC_2024 column
print(gdf["PrecC_2024"].value_counts(dropna=False))